# Stage 2: smoke test and full-run estimate

20 questions (10 BIRD in both evidence variants, 5 synthetic PL/EN pairs), the 2 cheapest available Bedrock
models and 1 local model (Qwen2.5-Coder-7B), strategies S0 and S2. Ends with the cost and time estimate for
stages 3 and 4 in `results/smoke_estimate.json`. **Review it before running stage 3.**

Prerequisites: `discover-models` run, prices filled in `config/prices.yaml`, BIRD fetched and sampled,
vLLM installed on the GPU machine.

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from t2sbench.bench import load_results
from t2sbench.report import aggregate
pd.set_option("display.max_colwidth", 120)
RUN = False  # set True to execute the stage from here (it can take hours and cost money)

def sh(cmd):
    print("$", cmd)
    if RUN:
        subprocess.run(cmd, shell=True, check=True)

def summary(stage):
    df = load_results(Path("results"), stage)
    if df.empty:
        print(f"no results for {stage} yet")
        return df
    agg = aggregate(df)
    return agg[["model", "adapter", "strategy", "dataset", "n", "ex", "ci_lo", "ci_hi", "syntax_ok", "no_answer",
                "time_median", "time_p90", "in_tokens_mean", "out_tokens_mean", "cost_per_100"]].sort_values(
                ["dataset", "ex"], ascending=[True, False])

In [ ]:
sh("uv run t2sbench discover-models")
if Path("results/model_availability.json").exists():
    display(pd.DataFrame(json.loads(Path("results/model_availability.json").read_text())))

In [ ]:
sh("uv run t2sbench stage 2")

In [ ]:
summary("smoke")

In [ ]:
est = Path("results/smoke_estimate.json")
if est.exists():
    e = json.loads(est.read_text())
    display(pd.Series(e["summary"]))
    display(pd.DataFrame(e["configs"]).groupby(["model", "strategy"])[["cost_usd", "hours"]].sum())